In [0]:
# ════════════════════════════════════════════
# Workshop Configuration — no edits needed
# ════════════════════════════════════════════
import re

# Read the current user directly from the Databricks workspace context
_raw_user   = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
USERNAME    = re.sub(r'[^a-zA-Z0-9]', '_', _raw_user.split("@")[0])

CATALOG     = "workshop"         # shared catalog, pre-configured by facilitator
SCHEMA      = USERNAME           # your personal schema inside the catalog

# MLflow experiment path — lives in your personal workspace folder
EXPERIMENT_PATH = f"/Users/{_raw_user}/gdm_yield_{USERNAME}"

print(f"📍 Logged in as  : {_raw_user}")
print(f"📍 USERNAME       : {USERNAME}")
print(f"📍 Your namespace : {CATALOG}.{SCHEMA}")
print(f"📍 Your table     : {CATALOG}.{SCHEMA}.yield_trials")
print(f"📍 Your model     : gdm-yield-model-{USERNAME}")
print(f"📍 MLflow path    : {EXPERIMENT_PATH}")

In [0]:
# Create your personal schema (run once)
# First, create the workshop catalog if it doesn't exist
spark.sql("CREATE CATALOG IF NOT EXISTS workshop")
print(f"✅ Catalog created: {CATALOG}")

# Now create the schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"✅ Schema ready: {CATALOG}.{SCHEMA}")

In [0]:
# ════════════════════════════════════════════
# Step 1: Download the Kaggle dataset
# ════════════════════════════════════════════

# Install kaggle CLI
!pip install kaggle --quiet

# Download the crop yield dataset from Kaggle
# Note: You need to have your Kaggle API credentials configured
# (kaggle.json in ~/.kaggle/ or set KAGGLE_USERNAME and KAGGLE_KEY environment variables)
!kaggle datasets download -d patelris/crop-yield-prediction-dataset

# Unzip the downloaded file
!unzip -o crop-yield-prediction-dataset.zip

print("✅ Dataset downloaded and extracted")

# ════════════════════════════════════════════
# Step 2: Load yield_df.csv into Spark DataFrame
# ════════════════════════════════════════════

df = spark.read.csv(
    "/Workspace/Users/nicolas.caceres5771@unaula.edu.co/Big_Data_Esp_UNAULA/yield_df.csv",
    header=True,
    inferSchema=True
)

print("✅ Dataset loaded into Spark DataFrame")

# ════════════════════════════════════════════
# Step 3: Explore the dataset
# ════════════════════════════════════════════

# Show schema (column names and types)
print("\n📋 SCHEMA:")
df.printSchema()

# Show total row count
total_rows = df.count()
print(f"\n📊 TOTAL ROWS: {total_rows:,}")

# Show 5 sample rows
print("\n📄 SAMPLE ROWS (5):")
df.show(5, truncate=False)

# Show unique crop types in the "Item" column
print("\n🌾 UNIQUE CROP TYPES (Item column):")
unique_items = df.select("Item").distinct().orderBy("Item")
print(f"Total unique crops: {unique_items.count()}")
unique_items.show(100, truncate=False)

# Show unique Area values that include "Argentina" or "Brazil"
print("\n🌎 AREAS INCLUDING 'Argentina' OR 'Brazil':")
from pyspark.sql.functions import col

areas_south_america = df.select("Area").distinct() \
    .filter(col("Area").contains("Argentina") | col("Area").contains("Brazil")) \
    .orderBy("Area")

print(f"Total matching areas: {areas_south_america.count()}")
areas_south_america.show(100, truncate=False)

In [0]:
# ════════════════════════════════════════════
# Step 4: Save DataFrame as Delta table
# ════════════════════════════════════════════

# Write the DataFrame to Delta format
df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.yield_trials")

print(f"✅ Delta table saved: {CATALOG}.{SCHEMA}.yield_trials")

# ════════════════════════════════════════════
# Step 5: Sanity checks
# ════════════════════════════════════════════

# Check 1: Count total rows
count_result = spark.sql(f"SELECT COUNT(*) as total_rows FROM {CATALOG}.{SCHEMA}.yield_trials")
print("\n📊 ROW COUNT:")
count_result.show()

# Check 2: Show sample rows
print("\n📄 SAMPLE ROWS (5):")
sample_result = spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.yield_trials LIMIT 5")
sample_result.show(truncate=False)

In [0]:
# ════════════════════════════════════════════
# Load data and build sklearn Pipeline with preprocessing
# ════════════════════════════════════════════

%pip install xgboost --quiet

import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split

# 1. Load the table into a pandas DataFrame
df_pandas = spark.table(f"{CATALOG}.{SCHEMA}.yield_trials").toPandas()

print(f"✅ Loaded table {CATALOG}.{SCHEMA}.yield_trials")
print(f"   Shape: {df_pandas.shape}")
print(f"   Columns: {list(df_pandas.columns)}")

# Define target and feature columns
target_col = "hg/ha_yield"
feature_cols = ["Item", "Area", "average_rain_fall_mm_per_year", 
                "pesticides_tonnes", "avg_temp", "Year"]

# Prepare X and y
X = df_pandas[feature_cols]
y = df_pandas[target_col]

print(f"\n📊 Target: {target_col}")
print(f"📊 Features: {feature_cols}")

# 2. Build sklearn Pipeline with ColumnTransformer
categorical_features = ["Item", "Area"]
numeric_features = ["average_rain_fall_mm_per_year", "pesticides_tonnes", "avg_temp", "Year"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42))
])

print("\n✅ Pipeline created with preprocessing and XGBRegressor")

# 3. Split the data 80/20 train/test with random_state=42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. Show the shapes
print("\n📐 Data split shapes:")
print(f"   X_train: {X_train.shape}")
print(f"   X_test:  {X_test.shape}")
print(f"   y_train: {y_train.shape}")
print(f"   y_test:  {y_test.shape}")

print("\n✅ Ready to train the pipeline!")

In [0]:
# ════════════════════════════════════════════
# Train the pipeline and log with MLflow
# ════════════════════════════════════════════

import mlflow
from mlflow.models import infer_signature
from sklearn.metrics import mean_squared_error, r2_score
import plotly.graph_objects as go
import numpy as np

# 1. Set MLflow experiment
mlflow.set_experiment(EXPERIMENT_PATH)
print(f"✅ MLflow experiment set to: {EXPERIMENT_PATH}")

# 2. Fit the pipeline
print("\n🚀 Training the pipeline...")
pipeline.fit(X_train, y_train)
print("✅ Pipeline trained successfully")

# 3. Evaluate on test set
y_pred = pipeline.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"\n📊 Test Set Performance:")
print(f"   RMSE: {rmse:.2f}")
print(f"   R²:   {r2:.4f}")

# 4. Log model with MLflow
with mlflow.start_run():
    # Log parameters
    mlflow.log_params({
        "n_estimators": 100,
        "max_depth": 5,
        "learning_rate": 0.1
    })
    
    # Log metrics
    mlflow.log_metrics({
        "rmse": rmse,
        "r2": r2
    })
    
    # Log the full pipeline with signature
    signature = infer_signature(X_train, y_pred)
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="gdm_yield_pipeline",
        registered_model_name=f"{CATALOG}.{SCHEMA}.gdm-yield-model-{USERNAME}",
        signature=signature
    )
    
    print(f"\n✅ Model logged to MLflow")
    print(f"   Registered as: {CATALOG}.{SCHEMA}.gdm-yield-model-{USERNAME}")

# 5. Create visualizations

# a. Feature importances bar chart
feature_importances = pipeline.named_steps["regressor"].feature_importances_
feature_names = ["Crop Type", "Country", "Rainfall (mm/yr)", "Pesticides (t)", "Avg Temp (°C)", "Year"]

# Sort by importance (descending)
sorted_idx = np.argsort(feature_importances)[::-1]
sorted_importances = feature_importances[sorted_idx]
sorted_names = [feature_names[i] for i in sorted_idx]

fig1 = go.Figure(go.Bar(
    x=sorted_importances,
    y=sorted_names,
    orientation='h',
    marker=dict(color='steelblue')
))

fig1.update_layout(
    title="Feature Importances (XGBoost)",
    xaxis_title="Importance",
    yaxis_title="Feature",
    height=400,
    yaxis=dict(autorange="reversed")  # Highest importance at top
)

print("\n📊 Feature Importances Chart:")
fig1.show()

# b. Predicted vs Actual scatter plot
fig2 = go.Figure()

# Add scatter points
fig2.add_trace(go.Scatter(
    x=y_test,
    y=y_pred,
    mode='markers',
    marker=dict(color='steelblue', size=6, opacity=0.6),
    name='Predictions'
))

# Add perfect prediction diagonal line
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
fig2.add_trace(go.Scatter(
    x=[min_val, max_val],
    y=[min_val, max_val],
    mode='lines',
    line=dict(color='red', dash='dash'),
    name='Perfect Prediction'
))

fig2.update_layout(
    title=f"Predicted vs Actual Yield (RMSE: {rmse:.2f}, R²: {r2:.4f})",
    xaxis_title="Actual Yield (hg/ha)",
    yaxis_title="Predicted Yield (hg/ha)",
    height=500,
    showlegend=True
)

print("\n📊 Predicted vs Actual Chart:")
fig2.show()

print("\n✅ Training and logging complete!")

In [0]:
# ════════════════════════════════════════════
# Make a prediction using the trained pipeline
# ════════════════════════════════════════════

import pandas as pd

# Define the input data
input_data = {
    "Item": "Maize",
    "Area": "Argentina",
    "average_rain_fall_mm_per_year": 750.0,
    "pesticides_tonnes": 32000.0,
    "avg_temp": 18.5,
    "Year": 2020
}

# Convert to DataFrame
input_df = pd.DataFrame([input_data])

print("📊 Input data:")
print(input_df)

# Make prediction using the trained pipeline
prediction = pipeline.predict(input_df)

print(f"\n✅ Predicted yield: {prediction[0]:.2f} hg/ha")
print(f"   (approximately {prediction[0]/10:.2f} tonnes/ha)")